# Proyecto TAM: Teoría de Aprendizaje de Máquinas
**Integrantes:** Jair Telpiz & Nathalia Mora  
**Estructura del Proyecto en 5 Secciones:**
1. **Sección 1:** Clasificación MNIST con Red Densa (DNN)
2. **Sección 2:** Clasificación MNIST con Red Convolucional (CNN 2D) y Mapas de Activación
3. **Sección 3:** Regresión No Lineal (Aproximación de Función Coseno - 1000 épocas)
4. **Sección 4:** Series de Tiempo Complejas (Comparación DNN, CNN1D, RNN, LSTM, GRU)
5. **Sección 5:** Agente RAG Especializado en Circuitos y Dispositivos (LangChain + Gemini API + ChromaDB)


## 0. Instalación e Importación de Dependencias Generales


In [ ]:
# Instalación de librerías con versiones compatibles (LangChain 0.3.x como en el taller del profe)
%pip install -q --upgrade-strategy only-if-needed \
    "langchain==0.3.27" \
    "langchain-core==0.3.80" \
    "langchain-community==0.3.27" \
    "langchain-google-genai==2.1.8" \
    "langchain-chroma==0.2.5" \
    "langchain-text-splitters==0.3.9" \
    "chromadb>=1.0.9,<2.0.0" \
    "pypdf>=5.0.0,<7.0.0" \
    "tensorflow" "keras" "matplotlib" "numpy" "pandas" "scikit-learn"


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Fijar semillas para reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)
print('TensorFlow Version:', tf.__version__)


---
# SECCIÓN 1: Clasificación MNIST con Red Densa (DNN)
En esta sección cargamos el dataset MNIST, aplanamos las imágenes a vectores de 784 características y entrenamos una red neuronal densa (Fully Connected).


In [ ]:
# 1.1 Carga y normalización de MNIST para Red Densa
(x_train_mnist, y_train_mnist), (x_test_mnist, y_test_mnist) = tf.keras.datasets.mnist.load_data()

# Normalización [0, 1] y aplanado a (784,)
x_train_dense = (x_train_mnist / 255.0).reshape(-1, 784).astype('float32')
x_test_dense = (x_test_mnist / 255.0).reshape(-1, 784).astype('float32')

print('x_train_dense shape:', x_train_dense.shape)
print('x_test_dense shape:', x_test_dense.shape)


In [ ]:
# 1.2 Definición del Modelo Funcional Densa
in_dense = tf.keras.layers.Input(shape=(784,), name='input_dense')
h1 = tf.keras.layers.Dense(128, activation='relu', name='densa_1')(in_dense)
h2 = tf.keras.layers.Dense(64, activation='relu', name='densa_2')(h1)
out_dense = tf.keras.layers.Dense(10, activation='softmax', name='output_dense')(h2)

model_dense = tf.keras.Model(inputs=in_dense, outputs=out_dense, name='MNIST_DNN')
model_dense.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_dense.summary()


In [ ]:
# 1.3 Entrenamiento y Guardado del Modelo Densa
history_dense = model_dense.fit(
    x_train_dense, y_train_mnist,
    epochs=15,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

loss_d, acc_d = model_dense.evaluate(x_test_dense, y_test_mnist, verbose=0)
print(f'Test Accuracy (Densa): {acc_d * 100:.2f}%')

# Guardar pesos para Streamlit
model_dense.save('mnist_model.keras')
print('Modelo guardado como mnist_model.keras')


---
# SECCIÓN 2: Clasificación MNIST con Red Convolucional (CNN 2D) y Mapas de Activación
Utilizamos capas Conv2D y MaxPooling2D para extraer características espaciales de los dígitos y visualizamos las activaciones intermedias.


In [ ]:
# 2.1 Preparación de datos para Convolucional (28, 28, 1)
x_train_conv = (x_train_mnist / 255.0).reshape(-1, 28, 28, 1).astype('float32')
x_test_conv = (x_test_mnist / 255.0).reshape(-1, 28, 28, 1).astype('float32')

# 2.2 Definición del Modelo CNN Funcional
in_conv = tf.keras.layers.Input(shape=(28, 28, 1), name='input_conv')
c1 = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', strides=1, padding='same', name='conv1')(in_conv)
p1 = tf.keras.layers.MaxPooling2D((2, 2), name='maxpool1')(c1)
c2 = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', strides=1, padding='same', name='conv2')(p1)
p2 = tf.keras.layers.MaxPooling2D((2, 2), name='maxpool2')(c2)

flat = tf.keras.layers.Flatten(name='flatten')(p2)
d_c = tf.keras.layers.Dense(64, activation='relu', name='dense_c')(flat)
out_conv = tf.keras.layers.Dense(10, activation='softmax', name='output_conv')(d_c)

model_conv = tf.keras.Model(inputs=in_conv, outputs=out_conv, name='MNIST_CNN')
model_conv.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_conv.summary()


In [ ]:
# 2.3 Entrenamiento y Guardado del Modelo Convolucional
history_conv = model_conv.fit(
    x_train_conv, y_train_mnist,
    epochs=15,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

loss_c, acc_c = model_conv.evaluate(x_test_conv, y_test_mnist, verbose=0)
print(f'Test Accuracy (Convolucional): {acc_c * 100:.2f}%')

# Guardar pesos para Streamlit
model_conv.save('mnist_model_conv.keras')
print('Modelo guardado como mnist_model_conv.keras')


In [ ]:
# 2.4 Extracción y Visualización de Feature Maps (Activaciones conv1 y conv2)
sample_idx = 0
sample_img = x_test_conv[sample_idx:sample_idx+1]

# Modelos intermedios para inspeccionar activación
model_act_c1 = tf.keras.Model(inputs=model_conv.input, outputs=model_conv.get_layer('conv1').output)
model_act_c2 = tf.keras.Model(inputs=model_conv.input, outputs=model_conv.get_layer('conv2').output)

act_c1 = model_act_c1.predict(sample_img)
act_c2 = model_act_c2.predict(sample_img)

fig, axes = plt.subplots(1, 9, figsize=(18, 3))
axes[0].imshow(sample_img[0, :, :, 0], cmap='gray')
axes[0].set_title(f'Entrada (Dígito: {y_test_mnist[sample_idx]})')
axes[0].axis('off')

# Primeros 4 canales de conv1
for i in range(4):
    axes[i+1].imshow(act_c1[0, :, :, i], cmap='viridis')
    axes[i+1].set_title(f'Conv1 - Filtro {i+1}')
    axes[i+1].axis('off')

# Primeros 4 canales de conv2
for i in range(4):
    axes[i+5].imshow(act_c2[0, :, :, i], cmap='magma')
    axes[i+5].set_title(f'Conv2 - Filtro {i+1}')
    axes[i+5].axis('off')

plt.tight_layout()
plt.show()


---
# SECCIÓN 3: Regresión No Lineal (Aproximación de Función Coseno)
En este módulo abordamos el problema de **regresión continua**: a partir de una entrada escalar $x$, una red neuronal profunda con activaciones `swish`, `relu` y `tanh` aprende a predecir el valor continuo de la onda $y = \cos(2\pi f x)$ utilizando función de pérdida **MSE** y **1000 épocas** de entrenamiento.


In [ ]:
# 3.1 Generación de la señal coseno y partición aleatoria (80% train / 20% test)
f = 2
fs = 100
x_cos = np.arange(-2, 2, 1/fs).reshape(-1, 1)
y_cos = np.cos(2 * np.pi * f * x_cos)

xtrain_cos, xtest_cos, ytrain_cos, ytest_cos = train_test_split(
    x_cos, y_cos, test_size=0.2, random_state=42, shuffle=True
)

plt.figure(figsize=(10, 4))
plt.plot(x_cos, y_cos, color='lightgray', linestyle='--', label='Señal completa')
plt.scatter(xtrain_cos, ytrain_cos, color='orange', s=10, label='Entrenamiento (80%)')
plt.scatter(xtest_cos, ytest_cos, color='green', s=10, label='Prueba (20%)')
plt.title('Partición de Datos para Regresión (Coseno)')
plt.xlabel('x')
plt.ylabel('Amplitud')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# 3.2 Construcción de la Red Profunda para Regresión del Coseno
inputs_cos = tf.keras.layers.Input(shape=(1,), name='input_cos')

d1 = tf.keras.layers.Dense(10, activation='swish', name='d1')(inputs_cos)
d2 = tf.keras.layers.Dense(32, activation='relu', name='d2')(d1)
d3 = tf.keras.layers.Dense(64, activation='relu', name='d3')(d2)
d4 = tf.keras.layers.Dense(32, activation='swish', name='d4')(d3)
d5 = tf.keras.layers.Dense(32, activation='tanh', name='d5')(d4)

# Salida lineal para regresión continua
outputs_cos = tf.keras.layers.Dense(1, activation='linear', name='output_cos')(d5)

model_cos = tf.keras.Model(inputs=inputs_cos, outputs=outputs_cos, name='modelo_coseno_profundo')
model_cos.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss='mse',
    metrics=['mae']
)
model_cos.summary()


In [ ]:
# 3.3 Entrenamiento (1000 épocas) y Guardado del Modelo
history_cos = model_cos.fit(
    xtrain_cos, ytrain_cos,
    epochs=1000,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

loss_cos, mae_cos = model_cos.evaluate(xtest_cos, ytest_cos, verbose=0)
print(f'Regresión Coseno - Test MSE: {loss_cos:.6f} | Test MAE: {mae_cos:.6f}')

model_cos.save('modelo_coseno_regresion.keras')
print('Modelo guardado como modelo_coseno_regresion.keras')

# 3.4 Visualización: Señal Real vs Predicción del Modelo
y_pred_cos = model_cos.predict(xtest_cos)
sort_idx = np.argsort(xtest_cos.flatten())

plt.figure(figsize=(10, 5))
plt.plot(x_cos, y_cos, color='lightgray', linestyle='--', label='Señal Coseno Real Completa')
plt.scatter(xtest_cos[sort_idx], ytest_cos[sort_idx], color='green', s=15, label='Datos de Prueba Reales (20%)')
plt.plot(xtest_cos[sort_idx], y_pred_cos[sort_idx], color='red', linewidth=2.5, label='Predicción del Modelo')
plt.title('Regresión: Señal Real vs Predicción del Modelo (Test)')
plt.xlabel('x')
plt.ylabel('Amplitud')
plt.legend()
plt.grid(True)
plt.show()


---
# SECCIÓN 4: Series de Tiempo Complejas (Comparación de Modelos)
Evaluamos y comparamos el desempeño de 5 arquitecturas:  
**DNN, CNN 1D, SimpleRNN, LSTM y GRU**  
sobre una señal que asemeja una serie temporal realista: **Tendencia + Dos Estacionalidades + Ruido Gaussiano**.


In [ ]:
# 4.1 Generación de la Serie de Tiempo Compleja
np.random.seed(42)
t_complex = np.arange(0, 400, 0.01) # 40,000 puntos

trend = 0.005 * t_complex
seasonality_1 = np.sin(2 * np.pi * 0.1 * t_complex)
seasonality_2 = 0.5 * np.cos(2 * np.pi * 0.5 * t_complex)
noise = np.random.normal(0, 0.15, size=t_complex.shape)

y_complex = trend + seasonality_1 + seasonality_2 + noise
# Normalización estandarizada
y_complex = (y_complex - np.mean(y_complex)) / np.std(y_complex)

# Ventaneo
w_comp = 1000
p_comp = 200

x_all_c, y_all_c = [], []
for i in range(0, y_complex.shape[0] - w_comp - p_comp, p_comp):
    x_all_c.append(y_complex[i:i+w_comp])
    y_all_c.append(y_complex[i+w_comp:i+w_comp+p_comp])

x_all_c = np.array(x_all_c)
y_all_c = np.array(y_all_c)

x_tr_c, x_te_c, y_tr_c, y_te_c = train_test_split(
    x_all_c, y_all_c, test_size=0.2, random_state=42, shuffle=True
)

x_tr_seq = x_tr_c[..., np.newaxis]
x_te_seq = x_te_c[..., np.newaxis]

print('Entrenamiento Complejo:', x_tr_c.shape)
print('Prueba Compleja:', x_te_c.shape)


In [ ]:
# 4.2 Definición de los 5 Modelos Comparativos
# 1. DNN
in_dnn = tf.keras.layers.Input(shape=(w_comp,))
d1 = tf.keras.layers.Dense(128, activation='relu')(in_dnn)
d2 = tf.keras.layers.Dense(64, activation='relu')(d1)
out_dnn = tf.keras.layers.Dense(p_comp, activation='linear')(d2)
model_dnn = tf.keras.Model(inputs=in_dnn, outputs=out_dnn, name='DNN_Compleja')

# 2. CNN 1D
in_cnn = tf.keras.layers.Input(shape=(w_comp, 1))
c1 = tf.keras.layers.Conv1D(32, 5, activation='relu', padding='same')(in_cnn)
p1 = tf.keras.layers.MaxPooling1D(2)(c1)
c2 = tf.keras.layers.Conv1D(64, 5, activation='relu', padding='same')(p1)
f = tf.keras.layers.Flatten()(c2)
d = tf.keras.layers.Dense(64, activation='relu')(f)
out_cnn = tf.keras.layers.Dense(p_comp, activation='linear')(d)
model_cnn = tf.keras.Model(inputs=in_cnn, outputs=out_cnn, name='CNN1D_Compleja')

# 3. SimpleRNN
in_rnn = tf.keras.layers.Input(shape=(w_comp, 1))
r1 = tf.keras.layers.SimpleRNN(64, activation='tanh')(in_rnn)
out_rnn = tf.keras.layers.Dense(p_comp, activation='linear')(r1)
model_rnn = tf.keras.Model(inputs=in_rnn, outputs=out_rnn, name='RNN_Compleja')

# 4. LSTM
in_lstm = tf.keras.layers.Input(shape=(w_comp, 1))
l1 = tf.keras.layers.LSTM(64, activation='tanh')(in_lstm)
out_lstm = tf.keras.layers.Dense(p_comp, activation='linear')(l1)
model_lstm = tf.keras.Model(inputs=in_lstm, outputs=out_lstm, name='LSTM_Compleja')

# 5. GRU
in_gru = tf.keras.layers.Input(shape=(w_comp, 1))
g1 = tf.keras.layers.GRU(64, activation='tanh')(in_gru)
out_gru = tf.keras.layers.Dense(p_comp, activation='linear')(g1)
model_gru = tf.keras.Model(inputs=in_gru, outputs=out_gru, name='GRU_Compleja')

model_list = [model_dnn, model_cnn, model_rnn, model_lstm, model_gru]
for m in model_list:
    m.compile(optimizer='adam', loss='mse', metrics=['mae'])


In [ ]:
# 4.3 Entrenamiento y Guardado de los Modelos
history_dict = {}
epochs = 15
batch_size = 32

for m in model_list:
    print(f'Entrenando {m.name}...')
    if 'DNN' in m.name:
        h = m.fit(x_tr_c, y_tr_c, epochs=epochs, batch_size=batch_size, validation_data=(x_te_c, y_te_c), verbose=0)
    else:
        h = m.fit(x_tr_seq, y_tr_c, epochs=epochs, batch_size=batch_size, validation_data=(x_te_seq, y_te_c), verbose=0)
    history_dict[m.name] = h.history
    
    # Guardar cada modelo para Streamlit
    m.save(f'{m.name.lower()}.keras')
    print(f'Guardado: {m.name.lower()}.keras')


In [ ]:
# 4.4 Gráficas Comparativas: Predicción Directa vs Recursiva
idx_sample = 0
input_seq = x_te_c[idx_sample]
target_seq = y_te_c[idx_sample]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
t_in = np.arange(0, w_comp)
t_pred = np.arange(w_comp, w_comp + p_comp)

# Gráfica 1: Predicción Directa
axes[0].plot(t_in, input_seq, label='Entrada Test (1000 datos)', color='black', alpha=0.7)
axes[0].plot(t_pred, target_seq, label='Real Esperado (200 datos)', color='green', linewidth=2.5, linestyle='--')

for m in model_list:
    if 'DNN' in m.name:
        pred = m.predict(input_seq.reshape(1, w_comp), verbose=0)[0]
    else:
        pred = m.predict(input_seq.reshape(1, w_comp, 1), verbose=0)[0]
    axes[0].plot(t_pred, pred, label=f'Predicción {m.name}', alpha=0.85)

axes[0].set_title('Predicción Directa (Evolución Dinámica)')
axes[0].set_xlabel('Pasos Temporales')
axes[0].set_ylabel('Valor Normalizado')
axes[0].legend()
axes[0].grid(True)

# Gráfica 2: Simulación Recursiva Paso a Paso
axes[1].plot(t_in, input_seq, label='Entrada Test', color='black', alpha=0.7)
axes[1].plot(t_pred, target_seq, label='Real Esperado', color='green', linewidth=2.5, linestyle='--')

for m in model_list:
    ventana = list(input_seq)
    pred_recursiva = []
    for _ in range(p_comp):
        arr = np.array(ventana[-w_comp:])
        if 'DNN' in m.name:
            out = m.predict(arr.reshape(1, w_comp), verbose=0)[0]
        else:
            out = m.predict(arr.reshape(1, w_comp, 1), verbose=0)[0]
        next_val = out[0]
        pred_recursiva.append(next_val)
        ventana.append(next_val)
    axes[1].plot(t_pred, pred_recursiva, label=f'Recursivo {m.name}', alpha=0.85)

axes[1].set_title('Simulación Autorregresiva Recursiva (Error Acumulado)')
axes[1].set_xlabel('Pasos Temporales')
axes[1].set_ylabel('Valor Normalizado')
axes[1].legend()
axes[1].grid(True)
plt.tight_layout()
plt.show()


---
# SECCIÓN 5: Agente RAG de Circuitos y Dispositivos Electrónicos
Implementación del agente de recuperación aumentada con generación (RAG) utilizando LangChain, Google Generative AI (Gemini) y ChromaDB.
El agente responde **exclusivamente sobre circuitos eléctricos y dispositivos electrónicos**, citando fuentes y rechazando preguntas fuera de dominio.


In [ ]:
# 5.1 Configuración de API Key (usando Colab Secrets o getpass como en el taller)
import os
from getpass import getpass

gemini_key = None
try:
    from google.colab import userdata
    for name in ('GOOGLE_API_KEY', 'GEMINI_API_KEY'):
        try:
            gemini_key = userdata.get(name)
            if gemini_key:
                break
        except Exception:
            pass
except Exception:
    gemini_key = os.environ.get('GOOGLE_API_KEY') or os.environ.get('GEMINI_API_KEY')

if not gemini_key:
    gemini_key = getpass('Ingresa tu API Key de Google AI Studio: ')

os.environ['GOOGLE_API_KEY'] = gemini_key
os.environ['GEMINI_API_KEY'] = gemini_key

PDF_PATH = 'Manual_Maestro_Circuitos_y_Dispositivos.pdf'
if os.path.exists(PDF_PATH):
    print(f'PDF encontrado: {PDF_PATH}')
else:
    print('Sube Manual_Maestro_Circuitos_y_Dispositivos.pdf a Colab para cargar el manual completo.')


In [ ]:
# 5.2 Carga del PDF, Fragmentación y Vectorización con LangChain + Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# Cargar y dividir el Manual Maestro en fragmentos
loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
split_docs = text_splitter.split_documents(docs)

# Embeddings y Vector Store en memoria
embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001', google_api_key=gemini_key)
chroma_db = Chroma.from_documents(documents=split_docs, embedding=embeddings)
retriever = chroma_db.as_retriever(search_kwargs={'k': 4})
print(f'Base vectorial creada con {len(split_docs)} fragmentos del Manual Maestro.')


In [ ]:
# 5.3 Creación del RAG con Restricción Estricta de Dominio
llm = ChatGoogleGenerativeAI(
    model='gemini-3.5-flash-lite',
    google_api_key=gemini_key,
    temperature=0.1
)

system_prompt = """Eres un asistente experto exclusivamente en Circuitos Eléctricos y Dispositivos Electrónicos.
Instrucciones obligatorias:
1. Responde de forma técnica, precisa y didáctica utilizando ÚNICAMENTE el contexto provisto abajo.
2. Si la pregunta NO está relacionada con circuitos, leyes eléctricas o dispositivos electrónicos, responde exactamente: 'Lo siento, solo puedo responder preguntas relacionadas con Circuitos y Dispositivos Electrónicos.'
3. Si la respuesta no se encuentra en el contexto, indica que no cuentas con esa información en el manual.

Contexto:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human', '{input}')
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_circuitos = create_retrieval_chain(retriever, question_answer_chain)

# 5.4 Pruebas del Agente
def consultar_agente(pregunta):
    print(f'Pregunta: {pregunta}')
    res = rag_circuitos.invoke({'input': pregunta})
    print(f'Respuesta:\n{res["answer"]}\n' + '-'*60)

# Prueba 1: Pregunta dentro del dominio
consultar_agente('¿Cómo funciona un transistor BJT y cuáles son sus regiones de operación?')

# Prueba 2: Pregunta fuera del dominio (debe ser rechazada)
consultar_agente('¿Quién ganó el mundial de fútbol 2022?')
